In [1]:
import os
os.chdir('..')

In [ ]:
from src.models.architecture.wgan_gp import Discriminator, Generator
from src.training.trainer_wgan_gp import Trainer

import torch
from torch.utils.data import DataLoader
from torchvision.transforms import v2

In [ ]:
ds = torch.load("/kaggle/input/wgan-cwt/DistinctClasses/Scalogram/B/train.pt", weights_only=True)

transform = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),               
    v2.Normalize([.5], [.5])  
])
data = transform(ds["data"])

data_loader = DataLoader(data, shuffle=True, batch_size=64)

In [ ]:
from models.architecture.wgan_gp import Discriminator, Generator

d_model = Discriminator(in_channels=1)
g_model = Generator()

betas = (0.5, .9)
eps = 700
lr = 1e-4

d_optim = torch.optim.Adam(d_model.parameters(), lr=lr, betas=betas)
g_optim = torch.optim.Adam(g_model.parameters(), lr=lr, betas=betas)

trainer = Trainer(
    generator=g_model, discriminator=d_model,
    g_optim=g_optim, d_optim=d_optim,
    gp_w=10, critic_iters=20, logs=20,
    cuda=torch.cuda.is_available()
)

# Pass real_data=data and eval_every=50
trainer.train(data_loader, eps, save=True, eval_every=100, real_data=data)
trainer.save()